In [25]:
import numpy as np
import xarray as xr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

In [26]:
filename  = "assets/Voxel_Schneckenstein_II_10x10.nc"
fieldName = "Schneckenstein_II_Prediction_0"

In [27]:
# ── Daten laden ───────────────────────────────────────────────────────────────
ds = xr.open_dataset(filename)
data = ds[fieldName]
vol  = data.values
vol  = np.nan_to_num(vol, nan=-1).astype(int)

x_coords = data['x'].values
y_coords = data['y'].values
z_coords = data['z'].values
nz, ny, nx = vol.shape

In [28]:
# ── Gesteinsklassen & Koordinatengitter ───────────────────────────────────────
classes   = np.unique(vol)
classes   = classes[classes >= 0]
n_classes = len(classes)

palette = px.colors.qualitative.Plotly
def class_color(i):
    return palette[i % len(palette)]

X3, Y3, Z3 = np.meshgrid(x_coords, y_coords, z_coords, indexing='ij')
vol_xyz    = np.transpose(vol, (2, 1, 0))   # (nx, ny, nz)
k0         = nz // 2

In [29]:
# ── Subplots ──────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("3D Volumen", f"XY-Schnitt  z = {z_coords[k0]:.1f} m"),
    column_widths=[0.65, 0.35],
    horizontal_spacing=0.05,
)

# ── 3D-Scatter pro Klasse ─────────────────────────────────────────────────────
for i, cls in enumerate(classes):
    mask = vol_xyz == cls
    fig.add_trace(
        go.Scatter3d(
            x=X3[mask], y=Y3[mask], z=Z3[mask],
            mode='markers',
            marker=dict(size=4, color=class_color(i), opacity=0.4),
            name=f"Klasse {cls}",
            legendgroup=f"cls{cls}",
            showlegend=True,
        ),
        row=1, col=1,
    )

# ── Schnittebene (Surface) ────────────────────────────────────────────────────
surface_trace_idx = n_classes
Z_plane0 = np.full((ny, nx), z_coords[k0])

fig.add_trace(
    go.Surface(
        x=x_coords, y=y_coords, z=Z_plane0,
        opacity=0.25, showscale=False, colorscale="Greys",
        name="Schnittebene", showlegend=False,
    ),
    row=1, col=1,
)

# ── 2D-Heatmaps pro Klasse ────────────────────────────────────────────────────
heatmap_trace_start = n_classes + 1

def make_slice(k, cls):
    slc = vol[k, :, :].astype(object)
    slc[slc != cls] = None
    return slc.tolist()

for i, cls in enumerate(classes):
    fig.add_trace(
        go.Heatmap(
            x=x_coords, y=y_coords,
            z=make_slice(k0, cls),
            colorscale=[[0, class_color(i)], [1, class_color(i)]],
            zmin=int(cls), zmax=int(cls),
            showscale=False,
            name=f"Klasse {cls}",
            legendgroup=f"cls{cls}",
            showlegend=False,
        ),
        row=1, col=2,
    )

fig = go.Figure(fig)

# ── Slider ────────────────────────────────────────────────────────────────────
steps = []
for k in range(nz):
    z_val   = z_coords[k]
    Z_plane = np.full((ny, nx), z_val)
    new_z_heatmaps = [make_slice(k, cls) for cls in classes]
    trace_indices  = [surface_trace_idx] + list(
        range(heatmap_trace_start, heatmap_trace_start + n_classes)
    )
    steps.append(dict(
        method="update",
        args=[
            {"z": [Z_plane] + new_z_heatmaps},
            {"annotations[1].text": f"XY-Schnitt  z = {z_val:.1f} m"},
            trace_indices,
        ],
        label=f"{z_val:.1f}",
    ))

sliders = [dict(
    active=k0,
    currentvalue={"prefix": "z = ", "suffix": " m", "font": {"size": 13}},
    pad={"t": 30, "b": 10},
    len=0.65, x=0.0, y=0.0,
    steps=steps,
)]

x_range = [float(x_coords.min()), float(x_coords.max())]
y_range = [float(y_coords.min()), float(y_coords.max())]
z_range = [float(z_coords.min()), float(z_coords.max())]

fig.update_layout(
    width=1300, height=700,
    margin=dict(l=0, r=10, t=50, b=90),
    sliders=sliders,
    legend=dict(
        title="Gesteinsklassen<br><sup>(klick = toggle | 0-9 = Taste)</sup>",
        x=1.01, y=0.95, xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgrey", borderwidth=1,
        itemclick="toggle", itemdoubleclick="toggleothers",
    ),
    scene=dict(
        xaxis=dict(title="X (m)", range=x_range),
        yaxis=dict(title="Y (m)", range=y_range),
        zaxis=dict(title="Z (m)", range=z_range),
        aspectmode="cube",
        domain=dict(x=[0.0, 0.62], y=[0.0, 1.0]),
    ),
)
fig.update_xaxes(title_text="X (m)", row=1, col=2)
fig.update_yaxes(title_text="Y (m)", row=1, col=2)

# ── Figure als HTML mit Keyboard-Steuerung ausgeben ───────────────────────────
#
# Wir exportieren die Plotly-Figure als HTML-String und betten ein
# <script>-Block ein, der:
#   • ArrowUp / ArrowDown  → verschiebt den Slider um ±1 Ebene
#   • Tasten 0-9           → togglet die Sichtbarkeit von Gesteinsklasse 0-9
#
# Ein kleines HUD (oben links) zeigt jederzeit:
#   - aktuelle z-Tiefe
#   - welche Klassen sichtbar sind

import json

# Alle Slider-Daten als kompaktes JSON für das JS bereitstellen
z_labels = [s["label"] for s in steps]          # ["123.0", "133.0", ...]
# Pro Step: z-Werte für Surface + Heatmaps (nur die z-Arrays, als Listen)
# Achtung: Z_plane ist ein ndarray → in Liste konvertieren
slider_z_data = []
for k in range(nz):
    z_val   = float(z_coords[k])
    Z_plane = np.full((ny, nx), z_val).tolist()
    heatmaps = [make_slice(k, cls) for cls in classes]
    slider_z_data.append({"z_val": z_val, "surface": Z_plane, "heatmaps": heatmaps})

fig_html = fig.to_html(
    full_html=True,
    include_plotlyjs="cdn",
    div_id="plotly-geo",
    config={"scrollZoom": True},
)

# JavaScript-Block, der NACH dem plotly-div eingefügt wird
keyboard_js = f"""
<style>
#kb-hud {{
  position: fixed;
  top: 730px; left: 12px;
  background: rgba(20,20,30,0.82);
  color: #e8e8e8;
  font-family: 'JetBrains Mono', 'Fira Code', monospace;
  font-size: 12px;
  padding: 10px 14px;
  border-radius: 8px;
  border: 1px solid rgba(255,255,255,0.12);
  backdrop-filter: blur(4px);
  z-index: 9999;
  pointer-events: none;
  line-height: 1.8;
  min-width: 220px;
}}
#kb-hud .hud-title {{
  font-size: 10px;
  letter-spacing: 0.12em;
  text-transform: uppercase;
  color: #888;
  margin-bottom: 4px;
}}
#kb-hud .hud-row {{
  display: flex;
  justify-content: space-between;
  gap: 16px;
}}
#kb-hud .hud-key {{
  background: rgba(255,255,255,0.1);
  border-radius: 3px;
  padding: 0 5px;
  font-size: 11px;
  color: #adf;
}}
.cls-pill {{
  display: inline-block;
  padding: 1px 6px;
  border-radius: 4px;
  margin: 1px 2px;
  font-size: 11px;
  transition: opacity 0.15s;
}}
.cls-pill.hidden {{ opacity: 0.25; text-decoration: line-through; }}
</style>

<div id="kb-hud">
  <div class="hud-title">⌨ Tastatursteuerung</div>
  <div class="hud-row">
    <span><span class="hud-key">↑↓</span> Ebene</span>
    <span id="hud-z">z = – m</span>
  </div>
  <div class="hud-row">
    <span><span class="hud-key">0–9</span> Klasse</span>
  </div>
  <div id="hud-classes" style="margin-top:4px"></div>
</div>

<script>
(function() {{

  // ── Konfiguration aus Python ───────────────────────────────────────────────
  const N_Z        = {nz};                     // Anzahl z-Ebenen
  const N_CLASSES  = {n_classes};              // Anzahl Gesteinsklassen
  const CLASSES    = {json.dumps([int(c) for c in classes])};  // Klassen-IDs
  // Trace-Indizes (müssen zu Python passen!)
  const SURF_IDX   = {surface_trace_idx};      // Index des Surface-Trace
  const HM_START   = {heatmap_trace_start};    // Erster Heatmap-Trace
  // Pro Klasse: Scatter3d-Index (0…N_CLASSES-1) und Heatmap-Index
  // scatter für Klasse i → Trace i
  // heatmap für Klasse i → Trace HM_START + i

  const SLIDER_DATA = {json.dumps([
      {"z_val": float(z_coords[k]),
       "surface": np.full((ny, nx), float(z_coords[k])).tolist(),
       "heatmaps": [make_slice(k, cls) for cls in classes]}
      for k in range(nz)
  ])};

  // ── Zustand ────────────────────────────────────────────────────────────────
  let currentK     = {k0};                    // Aktuelle z-Ebene
  let classVisible = new Array(N_CLASSES).fill(true);  // Alle Klassen sichtbar

  // ── HUD initialisieren ─────────────────────────────────────────────────────
  const colors = {json.dumps([class_color(i) for i in range(n_classes)])};
  const hudClasses = document.getElementById('hud-classes');
  CLASSES.forEach((cls, i) => {{
    const pill = document.createElement('span');
    pill.className  = 'cls-pill';
    pill.id         = 'pill-' + i;
    pill.textContent = i + ':K' + cls;
    pill.style.background = colors[i] + '55';
    pill.style.border = '1px solid ' + colors[i];
    pill.style.color  = '#eee';
    hudClasses.appendChild(pill);
  }});

  function updateHudZ() {{
    const z = SLIDER_DATA[currentK].z_val;
    document.getElementById('hud-z').textContent =
      'z = ' + z.toFixed(1) + ' m  [' + (currentK+1) + '/' + N_Z + ']';
  }}
  updateHudZ();

  function updateHudClasses() {{
    classVisible.forEach((vis, i) => {{
      const pill = document.getElementById('pill-' + i);
      if (vis) pill.classList.remove('hidden');
      else     pill.classList.add('hidden');
    }});
  }}

  // ── Plotly-div finden ──────────────────────────────────────────────────────
  // Plotly-Div kann sich noch im Aufbau befinden → warten
  function getDiv() {{
    return document.getElementById('plotly-geo');
  }}

  // ── z-Ebene wechseln ──────────────────────────────────────────────────────
  function changeLevel(delta) {{
    const newK = Math.min(Math.max(currentK + delta, 0), N_Z - 1);
    if (newK === currentK) return;
    currentK = newK;

    const d = SLIDER_DATA[currentK];
    const div = getDiv();
    if (!div) return;

    // Surface-z updaten
    const surfUpdate = {{ z: [d.surface] }};
    Plotly.restyle(div, surfUpdate, [SURF_IDX]);

    // Alle Heatmap-Traces updaten
    const hmIndices = Array.from({{length: N_CLASSES}}, (_, i) => HM_START + i);
    Plotly.restyle(div, {{ z: d.heatmaps }}, hmIndices);

    // Slider-Position updaten
    Plotly.relayout(div, {{ 'sliders[0].active': currentK }});

    // Untertitel updaten
    Plotly.relayout(div, {{ 'annotations[1].text': 'XY-Schnitt  z = ' + d.z_val.toFixed(1) + ' m' }});

    updateHudZ();
  }}

  // ── Klassen-Sichtbarkeit togglen ──────────────────────────────────────────
  function toggleClass(i) {{
    if (i >= N_CLASSES) return;
    classVisible[i] = !classVisible[i];
    const vis = classVisible[i];
    const div = getDiv();
    if (!div) return;

    // Scatter3d-Trace (3D-View)
    Plotly.restyle(div, {{ visible: vis ? true : 'legendonly' }}, [i]);
    // Heatmap-Trace (2D-View)
    Plotly.restyle(div, {{ visible: vis ? true : false }}, [HM_START + i]);

    updateHudClasses();
  }}

  // ── Keyboard-Handler ──────────────────────────────────────────────────────
  document.addEventListener('keydown', function(e) {{
    // Nicht feuern wenn in einem Input-Feld
    const tag = document.activeElement.tagName;
    if (tag === 'INPUT' || tag === 'TEXTAREA') return;

    if (e.key === 'ArrowUp') {{
      e.preventDefault();
      changeLevel(+1);   // Höhere z-Ebene
    }} else if (e.key === 'ArrowDown') {{
      e.preventDefault();
      changeLevel(-1);   // Tiefere z-Ebene
    }} else if (e.key >= '0' && e.key <= '9') {{
      toggleClass(parseInt(e.key));
    }}
  }});

}})();
</script>
"""

# JavaScript vor dem schließenden </body>-Tag einfügen
final_html = fig_html.replace("</body>", keyboard_js + "\n</body>")

# Als HTML-Datei speichern (öffne im Browser für volle Keyboard-Unterstützung)
output_path = "voxel_keyboard.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"Gespeichert: {output_path}")
print()
print("Steuerung:")
print("  ↑ / ↓     → nächste / vorherige z-Ebene")
print("  0 – 9     → Sichtbarkeit von Gesteinsklasse 0–9 umschalten")
print()
print("Hinweis: Keyboard-Events funktionieren nur im Browser (nicht direkt im Notebook).")
print("Öffne die HTML-Datei mit: import webbrowser; webbrowser.open(output_path)")

Gespeichert: voxel_keyboard.html

Steuerung:
  ↑ / ↓     → nächste / vorherige z-Ebene
  0 – 9     → Sichtbarkeit von Gesteinsklasse 0–9 umschalten

Hinweis: Keyboard-Events funktionieren nur im Browser (nicht direkt im Notebook).
Öffne die HTML-Datei mit: import webbrowser; webbrowser.open(output_path)


In [30]:
# ── Optional: direkt im Browser öffnen ───────────────────────────────────────
import webbrowser, os
webbrowser.open("file://" + os.path.abspath(output_path))

True